In [1]:
import sys
from pathlib import Path

# Move up one level from 'notebooks' to the root directory and add it to sys.path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [2]:
from functools import partial

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from src.data.split import split_train_validation
from src.preprocessing.text import build_vocabulary
from src.deep_learning.dataset import (
    ToxicCommentDataset,
    collate_fn,
)
from src.deep_learning.model import ToxicCommentBiLSTM

In [3]:
# 1. Split the dataset.
train_df, val_df = split_train_validation()

# 2. Build vocabulary using training comments only.
vocab = build_vocabulary(
    train_df["comment_text"],
    min_freq=2,
    max_vocab_size=30_000,
)

# 3. Create training dataset.
train_dataset = ToxicCommentDataset(
    train_df,
    vocab,
    max_length=300,
)

# 4. Create validation dataset.
val_dataset = ToxicCommentDataset(
    val_df,
    vocab,
    max_length=300,
)

# 5. Create training DataLoader.
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

# 6. Create validation DataLoader.
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

In [4]:
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Vocabulary size:", len(vocab))

Training samples: 127656
Validation samples: 31915
Training batches: 3990
Validation batches: 998
Vocabulary size: 30000


In [5]:
# Get one training batch.
train_batch = next(iter(train_loader))

# Get one validation batch.
val_batch = next(iter(val_loader))

print("TRAINING BATCH")
print("Input IDs:", train_batch["input_ids"].shape)
print("Labels:", train_batch["labels"].shape)

print("\nVALIDATION BATCH")
print("Input IDs:", val_batch["input_ids"].shape)
print("Labels:", val_batch["labels"].shape)

TRAINING BATCH
Input IDs: torch.Size([32, 300])
Labels: torch.Size([32, 6])

VALIDATION BATCH
Input IDs: torch.Size([32, 300])
Labels: torch.Size([32, 6])


In [6]:
# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 2050


In [7]:
# Initialize Model, Loss, and Optimizer-

model = ToxicCommentBiLSTM(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_size=128,
    num_labels=6,
    dropout=0.3,
    pad_id=vocab.pad_id,
).to(device)

# Loss and optimizer.
criterion = nn.BCEWithLogitsLoss()

optimizer = Adam(
    model.parameters(),
    lr=0.001,
)

In [8]:
from src.deep_learning.trainer import train_one_epoch, evaluate

# Train for one epoch.
train_loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
)

print("Training loss:", train_loss)

# Evaluate on validation data.
val_loss = evaluate(
    model,
    val_loader,
    criterion,
    device,
)

print("Validation loss:", val_loss)

Training loss: 0.07632224527561471
Validation loss: 0.0566239133780697


In [9]:
# Experiment	Device	Epochs	Training loss	Validation loss	Time
# BiLSTM v1	    CPU	      1	      0.076618	     0.056350	12 min
# BiLSTM v1	    RTX 2050  1	      0.076628	     0.057901	48 sec

In [10]:
from src.deep_learning.trainer import predict
from src.evaluation.metrics import calculate_metrics

# Generate predictions using the validation dataset.
y_true, y_prob = predict(
    model=model,
    dataloader=val_loader,
    device=device,
)

# Calculate evaluation metrics.
results = calculate_metrics(
    y_true=y_true,
    y_prob=y_prob,
    threshold=0.5,
)

print("\nPer-label metrics:")
print(results["per_label"].to_string(index=False))

print("\nAggregate metrics:")
for metric, value in results["aggregate"].items():
    print(f"{metric}: {value:.4f}")

print("\nConfusion matrices:")
for label, matrix in results["confusion_matrices"].items():
    print(f"\n{label}:")
    print(matrix)



Per-label metrics:
        label  precision   recall       f1  support
        toxic   0.859450 0.644306 0.736488     3056
 severe_toxic   0.608108 0.140187 0.227848      321
      obscene   0.871917 0.639067 0.737550     1715
       threat   0.000000 0.000000 0.000000       74
       insult   0.803104 0.513011 0.626087     1614
identity_hate   0.000000 0.000000 0.000000      294

Aggregate metrics:
micro_precision: 0.8463
micro_recall: 0.5567
micro_f1: 0.6716
macro_f1: 0.3880
weighted_f1: 0.6502

Confusion matrices:

toxic:
[[28537   322]
 [ 1087  1969]]

severe_toxic:
[[31565    29]
 [  276    45]]

obscene:
[[30039   161]
 [  619  1096]]

threat:
[[31841     0]
 [   74     0]]

insult:
[[30098   203]
 [  786   828]]

identity_hate:
[[31621     0]
 [  294     0]]


In [11]:

import numpy as np

from src.data.loader import LABEL_COLUMNS


def inspect_probabilities(y_true, y_prob):
    y_true = np.asarray(y_true)
    y_prob = np.asarray(y_prob)

    thresholds = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]

    for i, label in enumerate(LABEL_COLUMNS):

        positive_probs = y_prob[y_true[:, i] == 1, i]
        negative_probs = y_prob[y_true[:, i] == 0, i]

        print(f"\n{'=' * 50}")
        print(f"Label: {label}")
        print(f"{'=' * 50}")

        print(f"Actual positive samples: {len(positive_probs)}")

        print("\nProbabilities for actual positive samples:")
        print(f"  Mean:   {positive_probs.mean():.4f}")
        print(f"  Median: {np.median(positive_probs):.4f}")
        print(f"  Maximum: {positive_probs.max():.4f}")

        print("\nPredicted positive samples at different thresholds:")

        for threshold in thresholds:
            predicted_count = np.sum(y_prob[:, i] >= threshold)
            print(
                f"  Threshold {threshold:.2f}: "
                f"{predicted_count} predictions"
            )


inspect_probabilities(y_true, y_prob)


Label: toxic
Actual positive samples: 3056

Probabilities for actual positive samples:
  Mean:   0.6290
  Median: 0.7467
  Maximum: 0.9987

Predicted positive samples at different thresholds:
  Threshold 0.05: 7478 predictions
  Threshold 0.10: 5305 predictions
  Threshold 0.20: 3779 predictions
  Threshold 0.30: 3095 predictions
  Threshold 0.40: 2637 predictions
  Threshold 0.50: 2291 predictions

Label: severe_toxic
Actual positive samples: 321

Probabilities for actual positive samples:
  Mean:   0.2572
  Median: 0.2343
  Maximum: 0.7768

Predicted positive samples at different thresholds:
  Threshold 0.05: 985 predictions
  Threshold 0.10: 686 predictions
  Threshold 0.20: 427 predictions
  Threshold 0.30: 257 predictions
  Threshold 0.40: 156 predictions
  Threshold 0.50: 74 predictions

Label: obscene
Actual positive samples: 1715

Probabilities for actual positive samples:
  Mean:   0.6239
  Median: 0.7634
  Maximum: 0.9963

Predicted positive samples at different thresholds:


In [13]:
from sklearn.metrics import f1_score

thresholds = [0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50]

for i, label in enumerate(LABEL_COLUMNS):

    print(f"\n{label}")

    for threshold in thresholds:
        # Changed .astype(int) to .int()
        predictions = (y_prob[:, i] >= threshold).int()

        score = f1_score(
            y_true[:, i],
            predictions,
            zero_division=0,
        )

        print(f"Threshold: {threshold:.2f} | F1: {score:.4f}")



toxic
Threshold: 0.05 | F1: 0.5415
Threshold: 0.10 | F1: 0.6449
Threshold: 0.15 | F1: 0.6950
Threshold: 0.20 | F1: 0.7204
Threshold: 0.30 | F1: 0.7465
Threshold: 0.40 | F1: 0.7465
Threshold: 0.50 | F1: 0.7365

severe_toxic
Threshold: 0.05 | F1: 0.3920
Threshold: 0.10 | F1: 0.4449
Threshold: 0.15 | F1: 0.4585
Threshold: 0.20 | F1: 0.4759
Threshold: 0.30 | F1: 0.4291
Threshold: 0.40 | F1: 0.3690
Threshold: 0.50 | F1: 0.2278

obscene
Threshold: 0.05 | F1: 0.6033
Threshold: 0.10 | F1: 0.6948
Threshold: 0.15 | F1: 0.7438
Threshold: 0.20 | F1: 0.7635
Threshold: 0.30 | F1: 0.7694
Threshold: 0.40 | F1: 0.7567
Threshold: 0.50 | F1: 0.7376

threat
Threshold: 0.05 | F1: 0.0962
Threshold: 0.10 | F1: 0.0000
Threshold: 0.15 | F1: 0.0000
Threshold: 0.20 | F1: 0.0000
Threshold: 0.30 | F1: 0.0000
Threshold: 0.40 | F1: 0.0000
Threshold: 0.50 | F1: 0.0000

insult
Threshold: 0.05 | F1: 0.5477
Threshold: 0.10 | F1: 0.6352
Threshold: 0.15 | F1: 0.6699
Threshold: 0.20 | F1: 0.6872
Threshold: 0.30 | F1: 0.68